In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/supplement-extra"
setup_plotting(figure_dir, display_dpi=200, save_dpi=300)

import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

In [ ]:
def _sample_sort_key(x: str) -> tuple[int, int]:
    m = re.fullmatch(r"([CS])(\d+)", str(x))
    if not m:
        return (2, 10**12)  # put anything unexpected at the end
    grp, num = m.group(1), int(m.group(2))
    return (0 if grp == "C" else 1, num)

## Analyze batch mixing for all cell types

In [ ]:
def plot_counts_stacked(
    df_counts,
    figsize=(16, 6),
    save_path=None,
    title=None,
    expected_cols=None,
    col_name_mapping=None,
    ncols=8,
    colormap="tab20",
    ylabel="Sample",
    legend_bbox_to_anchor=(0.5, -0.05),
):
    # If normalize is True, create stacked bar plot with absolute counts on the right
    expected_cols = df_counts.columns.tolist() or expected_cols
    col_name_mapping = col_name_mapping or {c: c for c in expected_cols}
    # Ensure we have the expected columns
    for col in expected_cols:
        if col not in df_counts.columns:
            raise ValueError(f"Expected column '{col}' not found in df_counts")

    # Normalize each row to sum to 1 for proportions
    df_plot = df_counts[expected_cols].div(df_counts[expected_cols].sum(axis=1), axis=0)

    # Calculate total counts per sample
    total_counts = df_counts[expected_cols].sum(axis=1)

    # Define column names and corresponding labels/colors
    plot_configs = [(col, col_name_mapping[col]) for col in expected_cols]

    # Get samples
    samples = df_plot.index.tolist()
    y_pos = np.arange(len(samples))

    # Create figure with 2 subplots side by side, right axes 20% width of left
    fig, axes = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"width_ratios": [5, 1]}
    )
    fig.subplots_adjust(wspace=0.3)

    ax_left = axes[0]
    ax_right = axes[1]

    # Colors for each category
    colors = sns.color_palette(colormap, n_colors=len(plot_configs))

    # Left subplot: stacked horizontal bars (proportions)
    left = np.zeros(len(samples))
    for idx, (col, label) in enumerate(plot_configs):
        values = df_plot[col].values
        ax_left.barh(
            y_pos, values, left=left, align="center", label=label, color=colors[idx]
        )
        left += values

    # Set y-axis ticks and labels
    ax_left.set_yticks(y_pos)
    ax_left.set_yticklabels(samples, fontsize=8)
    ax_left.set_ylabel(ylabel, fontsize=10)
    ax_left.tick_params(axis="x", labelsize=8)

    # Set x-axis label and limits
    ax_left.set_xlabel("Proportion of cells", fontsize=10)
    ax_left.set_xlim(0, 1)

    # Add legend below the plot
    handles, labels = ax_left.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=legend_bbox_to_anchor,
        ncol=ncols,
        fontsize=10,
        columnspacing=0.6,  # Reduce column spacing (default is 2.0)
    )

    # Add grid for better readability
    ax_left.grid(axis="x", alpha=0.3, linestyle="--")
    sns.despine(top=True, right=True, ax=ax_left)

    # Right subplot: absolute counts
    muted_red = "#c44e52"  # Muted red color
    ax_right.barh(y_pos, total_counts.values, align="center", color=muted_red)
    ax_right.set_yticks(y_pos)
    ax_right.set_yticklabels(samples, fontsize=8)
    ax_right.set_xlabel("Number of cells", fontsize=10)
    ax_right.set_title("Total cells", fontsize=10, fontweight="bold")
    ax_right.grid(axis="x", alpha=0.3, linestyle="--")
    sns.despine(top=True, right=True, ax=ax_right)

    # Add overall title if provided
    if title is not None:
        fig.suptitle(title, fontsize=10, fontweight="bold", y=1.02)

    plt.tight_layout(rect=[0, 0.1, 1, 1])

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    return fig, axes

In [ ]:
path = "data/xenium/processed/04.1-kidney_tcr_tsub_harmonized.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()
adata

In [ ]:
adata.obs["condition"].value_counts()

In [ ]:
adata.obs["slide_short"] = adata.obs["slide"].map(
    {s: f"Slide-{i}" for i, s in enumerate(adata.obs["slide"].unique())}
)
control_samples = adata.obs[adata.obs["condition"] == "Control"]["sample"].unique()
samples = list(adata.obs["sample"].unique())

controls = [s for s in samples if s in control_samples]
cases = [s for s in samples if s not in control_samples]

mapping = {s: f"C{i}" for i, s in enumerate(controls)}
mapping.update({s: f"S{i}" for i, s in enumerate(cases)})

adata.obs["sample_short"] = adata.obs["sample"].map(mapping)

# adata.obs["sample_short"] = adata.obs["sample"].map(
#     {
#         s: f"C{i}" if s in control_samples else f"S{i}"
#         for i, s in enumerate(adata.obs["sample"].unique())
#     }
# )
adata.obs["sample_short"].value_counts()

In [ ]:
counts = adata.obs.groupby(["sample_short"])["cell_type_l1"].value_counts().unstack()

counts = counts.loc[sorted(counts.index, key=_sample_sort_key)]
plot_counts_stacked(
    counts,
    colormap="tab20",
    figsize=(8, 4),
    save_path=f"{figure_dir}/cell_type_props_sample.pdf",
)

In [ ]:
counts = adata.obs.groupby(["slide_short"])["cell_type_l1"].value_counts().unstack()

counts = counts.loc[sorted(counts.index, key=_sample_sort_key)]
plot_counts_stacked(
    counts,
    figsize=(16, 2),
    legend_bbox_to_anchor=(0.5, -0.25),
    ylabel="Slide",
)

## Analyze batch mixing for tcell subtypes

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()

In [ ]:
counts = ad_t.obs.groupby(["sample_short"])["cell_type_l2"].value_counts().unstack()

counts = counts.loc[sorted(counts.index, key=_sample_sort_key)]
plot_counts_stacked(
    counts,
    # legend_bbox_to_anchor=(0.5, 0.03),
    ylabel="Sample",
    figsize=(8, 4),
    save_path=f"{figure_dir}/tcell_subtype_props_sample.pdf",
)

In [ ]:
counts = ad_t.obs.groupby(["slide_short"])["cell_type_l2"].value_counts().unstack()

counts = counts.loc[sorted(counts.index, key=_sample_sort_key)]
plot_counts_stacked(
    counts, figsize=(16, 2), legend_bbox_to_anchor=(0.5, -0.04), ylabel="Slide"
)

## Compare mixing with with and without Harmony integration for subtypes

In [ ]:
import scanpy.external as sce

In [ ]:
ad_t.X = ad_t.layers["counts"].copy()
sc.pp.normalize_total(ad_t)
sc.pp.log1p(ad_t)
sc.pp.highly_variable_genes(ad_t, n_top_genes=100, batch_key="sample")

In [ ]:
markers_flattend = []
reduced_var_names = ad_t.var_names[ad_t.var["highly_variable"]].tolist()
reduced_var_names = [
    v for v in reduced_var_names if v not in markers_flattend
] + markers_flattend
ad_t = ad_t[:, reduced_var_names].copy()
ad_t

In [ ]:
ad_t.X = ad_t.layers["counts"].copy()
sc.pp.normalize_total(ad_t)
sc.pp.log1p(ad_t)
sc.tl.pca(ad_t, n_comps=50)

In [ ]:
sce.pp.harmony_integrate(
    ad_t, "sample", adjusted_basis="X_pca_harmony_sample", max_iter_harmony=20
)
sc.pp.neighbors(ad_t, use_rep="X_pca_harmony_sample")
sc.tl.leiden(ad_t, resolution=1.0, key_added="leiden_harmony_sample")
ad_t.obs["leiden_harmony_sample"].value_counts()

In [ ]:
sce.pp.harmony_integrate(ad_t, "slide", adjusted_basis="X_pca_harmony_slide")
sc.pp.neighbors(ad_t, use_rep="X_pca_harmony_slide")
sc.tl.leiden(ad_t, resolution=1.0, key_added="leiden_harmony_slide")
ad_t.obs["leiden_harmony_slide"].value_counts()

In [ ]:
ad_t.obsm["X_pca_sample"] = ad_t.obsm["X_pca"].copy()
ad_t.obsm["X_pca_slide"] = ad_t.obsm["X_pca"].copy()

In [ ]:
from scib_metrics.benchmark import Benchmarker

batch_key = ["sample", "slide", "sample", "slide"]
embed_keys = [
    "X_pca_sample",
    "X_pca_slide",
    "X_pca_harmony_sample",
    "X_pca_harmony_slide",
]
label_keys = [
    "cell_type_l2",
    "cell_type_l2",
    "leiden_harmony_sample",
    "leiden_harmony_slide",
]
all_results = []
for bk, ek, lk in zip(batch_key, embed_keys, label_keys):
    bm = Benchmarker(
        ad_t,
        batch_key=bk,
        label_key=lk,  # or "leiden" / your cluster column
        embedding_obsm_keys=[ek],
    )
    bm.benchmark()
    results = bm.get_results()
    all_results.append(results)
    display(results)
    # break
all_results = pd.concat(all_results, axis=0)
all_results

In [ ]:
metrics = all_results[~all_results.index.str.contains("Metric")]
metrics = metrics.reset_index()
metrics.to_csv("data/xenium/processed/09-batch_mixing_metrics.csv", index=False)

In [ ]:
metrics = pd.read_csv("data/xenium/processed/09-batch_mixing_metrics.csv")
metrics

In [ ]:
metrics[["Embedding"]]

## Plot comparison

In [ ]:
# pick the three embeddings you want
embeddings = [
    "X_pca_sample",
    "X_pca_harmony_sample",
    "X_pca_slide",
    "X_pca_harmony_slide",
]

# mapping for cleaner legend names
embedding_names = {
    "X_pca_sample": "PCA (sample)",
    "X_pca_slide": "PCA (slide)",
    "X_pca_harmony_sample": "Harmony (sample)",
    "X_pca_harmony_slide": "Harmony (slide)",
}

# keep only iLISI + cLISI and reshape so x-axis = metric, hue = embedding
df = (
    metrics.loc[metrics["Embedding"].isin(embeddings), ["Embedding", "iLISI", "BRAS"]]
    .drop_duplicates("Embedding")
    .melt(id_vars="Embedding", var_name="Metric", value_name="Score")
)

fig, ax = plt.subplots(figsize=(4, 3))
ax = sns.barplot(
    data=df,
    x="Metric",
    y="Score",
    hue="Embedding",
    order=["iLISI", "BRAS"],  # optional: control x-axis order
    hue_order=embeddings,  # optional: control legend/bar order
    errorbar=None,
    ax=ax,
)

ax.set_ylim(0, 1)
ax.set_xlabel("batch mixing metric", fontsize=10)
ax.set_ylabel("score", fontsize=10)
legend = ax.legend(
    title="",
    frameon=False,
    ncol=1,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=10,
)
ax.tick_params(axis="both", labelsize=8)

# update legend labels with cleaner names
for t in legend.get_texts():
    t.set_text(embedding_names.get(t.get_text(), t.get_text()))
sns.despine()
plt.tight_layout()
plt.savefig(
    f"{figure_dir}/batch_mixing_metrics.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()